# ALM v12.0 - Google Colab GPU Execution

This notebook configures a Google Colab GPU instance to run the ALM cognitive pipeline.

### Instructions:
1. Go to **Runtime > Change runtime type** in the top menu.
2. Select **T4 GPU**, **L4 GPU**, or **A100 GPU** as the hardware accelerator.
3. Run the cells below sequentially.

## 1. Mount Google Drive
Upload your entire `alm-project` folder to your Google Drive, then run this cell to attach your Drive to this notebook.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Change directory to the ALM project folder in your Drive
# Update this path if you placed the folder somewhere else!
%cd /content/drive/MyDrive/alm-project

## 2. Install Dependencies
Google Colab instances reset every time you disconnect. You must install the required dependencies into the temporary environment.

In [ ]:
!pip install --upgrade pip
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install transformers accelerate faster-whisper librosa numpy pandas pydantic
!pip install httpx huggingface_hub

## 3. Verify GPU Connection
Ensure PyTorch can successfully see the NVIDIA GPU.

In [ ]:
import torch
print("GPU Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))

## 4. Hardware Optimization Hotfix (Optional)
If you previously modified `core_modules/feature_extractor.py` to use CPU/MPS, run this cell to switch it back to highly-optimized GPU precision.

In [ ]:
import os
filepath = 'core_modules/feature_extractor.py'
if os.path.exists(filepath):
    with open(filepath, 'r') as f:
        code = f.read()
    
    # Revert compute_precision for NVIDIA CUDA acceleration
    code = code.replace('compute_precision = "auto"', 'compute_precision = "float16"')
    
    with open(filepath, 'w') as f:
        f.write(code)
    print("Optimized feature_extractor.py for NVIDIA GPUs.")

## 5. Execute Evaluation Pipeline
Run the full evaluation pipeline. With an L4 or A100 GPU, processing all 50 samples should take a fraction of the time compared to a CPU!

In [ ]:
!python research/evaluation_runner.py